# Gear Spall Data Atlas: interactive figures (Plotly)

Builds every figure and table in the Atlas report from the flattened parquet tables in the team bucket.

**Run order:**
1. **Load data** reuses your `lf`, `particles` and `channels` if you've already loaded them, and adds the validation and test splits.
2. **Build tables** removes the duplicate Exp-A Run-2, flags steady-state seconds and makes the 1-minute table.
3. **Photo labels** reads `photo_proxy.csv` if it's in the bucket; otherwise it scores the tooth photos (slower, about 1 GB).
4. **Compute** builds `D`, the dict every figure cell reads.
5. **Plot setup**, then any figure cell in any order.

Colab already has everything this needs (pandas, gcsfs, scikit-learn, scipy, Pillow, Plotly).

## 1 · Load data

In [ ]:
# 1 · Load data: reuse lf / particles / channels if already loaded; add the validation and test splits
import pandas as pd, numpy as np
try:
    from google.colab import auth
    auth.authenticate_user()          # pick your @umich.edu account in the popup
except ImportError:
    pass                              # not in Colab: DATA_ROOT can be a local path

DATA_ROOT = "gs://siads696-phm2026/data"
SPLITS = ["train", "validation", "test"]

def _read(path, reader=pd.read_parquet):
    try:
        return reader(path)
    except (FileNotFoundError, OSError) as e:
        print(f"  skipped {path} ({type(e).__name__})"); return None

LF = {"train": lf} if "lf" in globals() else {}
PT = {"train": particles} if "particles" in globals() else {}
for s in SPLITS:
    if s not in LF: LF[s] = _read(f"{DATA_ROOT}/{s}/lf_1s.parquet")
    if s not in PT: PT[s] = _read(f"{DATA_ROOT}/{s}/particles.parquet")
LF = {s: v for s, v in LF.items() if v is not None}; PT = {s: v for s, v in PT.items() if v is not None}
if "channels" not in globals():
    channels = pd.read_csv(f"{DATA_ROOT}/channels.csv")
for s, v in LF.items():
    print(f"{s:<11} {len(v):>9,} one-second rows · gears {sorted(v.experiment.unique())}")

## 2 · Build 1-second and 1-minute tables

In [ ]:
# 2 · Build tables: drop the duplicate run, flag steady-state seconds, 1-minute medians
frames, parts = [], []
for s, x in LF.items():
    x = x.copy(); x["split"] = s
    # Exp-A Run-2 files are byte copies of Run-1: blank their values but keep the nominal hour axis for photo alignment
    dup = (x.experiment == "Exp-A") & (x.run_num == 2)
    x.loc[dup, [c for c in x if c.startswith(("ci_", "ctx_"))]] = np.nan
    x["minute"] = (x.test_hours * 60).astype(int)
    frames.append(x)
    p = PT[s].copy(); p["split"] = s
    parts.append(p[~((p.experiment == "Exp-A") & (p.run_num == 2))])
S1 = pd.concat(frames, ignore_index=True)
PART = pd.concat(parts, ignore_index=True)
CI  = [c for c in S1 if c.startswith("ci_")]
CTX = [c for c in S1 if c.startswith("ctx_")]

# steady state: within 20 rpm of 1,400 rpm and within 5% of the gear's nominal (median) torque
nom_t = S1.groupby("experiment").ctx_pau_torque.transform("median")
S1["steady"] = (S1.ctx_pau_speed.sub(1400).abs() < 20) & (S1.ctx_pau_torque.sub(nom_t).abs() < 0.05 * nom_t)
M1 = S1[S1.steady].groupby(["split", "experiment", "run_num", "minute"])[CI + CTX].median().reset_index()
M1["hours"] = M1.minute / 60
print(f"{len(S1):,} one-second rows · {S1.steady.mean():.1%} steady · {len(M1):,} one-minute rows")

## 3 · Photo labels

In [ ]:
# 3 · Photo labels: texture score per tooth-inspection + eye-checked initiation / spall labels
import re, io, fsspec
from PIL import Image, ImageFilter
from concurrent.futures import ThreadPoolExecutor

PHOTO_PROXY = "gs://siads696-phm2026/eda/photo_proxy.csv"   # copy of eda/out/photo_proxy.csv (fast path)
PHOTO_DIR   = f"{DATA_ROOT}/train/photos"                     # fallback: score the photos here (~1 GB)
ORDER = {"Break-In": -2, "Pre-Run": -1, "Test-Start": -1}

def parse_photo(name):
    m = re.match(r"(Exp-\w)_(Break-In|Test-Start|Pre-Run|Run-\d+)_(.*)\.jpg", name)
    if not m: return None
    e, insp, rest = m.groups(); t = re.findall(r"Tooth_?(\d+)", rest)
    if not t: return None
    return dict(experiment=e, inspection=insp, tooth=int(t[0]), file=name,
                insp_order=ORDER.get(insp, int(insp.split("-")[1]) if insp.startswith("Run") else 0))

def texture_score(img):
    """Share of high-gradient pixels on the flank below the contact band: large spalls score high."""
    a = np.asarray(img.convert("L").resize((320, 180)).filter(ImageFilter.GaussianBlur(1.2)), dtype=float)
    gy, gx = np.gradient(a); g = np.hypot(gx, gy)
    return float((g[92:160, 64:256] > 0.09 * np.median(a[92:160, 64:256])).mean())

PH = _read(PHOTO_PROXY, pd.read_csv)
if PH is None:
    fs, root = fsspec.core.url_to_fs(PHOTO_DIR)
    paths = fs.glob(root.rstrip("/") + "/*.jpg")
    if not paths:
        raise FileNotFoundError(f"No photo_proxy.csv at {PHOTO_PROXY} and no .jpg files under {PHOTO_DIR}")
    print(f"scoring {len(paths)} photos from {PHOTO_DIR} ...")
    def _one(p):
        d = parse_photo(p.split("/")[-1])
        if d:
            with fs.open(p, "rb") as f: d["tex"] = texture_score(Image.open(io.BytesIO(f.read())))
        return d
    with ThreadPoolExecutor(16) as ex:
        rows = [r for r in ex.map(_one, paths) if r]
    PH = (pd.DataFrame(rows).groupby(["experiment", "inspection", "insp_order", "tooth"], as_index=False)
            .agg(tex=("tex", "median"), n_photos=("file", "size")))
PH = PH[["experiment", "inspection", "insp_order", "tooth", "tex", "n_photos"]].copy()
PH["tooth"] = PH.tooth.astype(int)

# Checked by eye on each gear's last two inspections; everything else relies on the texture score
EYE = {("Exp-A",16,4):"init", ("Exp-A",28,4):"init", ("Exp-B",11,6):"init", ("Exp-B",14,6):"init",
       ("Exp-F",9,7):"init", ("Exp-F",12,7):"init", ("Exp-F",13,7):"init",
       **{k: "spall" for k in [("Exp-A",16,5), ("Exp-A",24,5), ("Exp-A",28,5), ("Exp-B",11,7), ("Exp-B",14,7),
                               ("Exp-C",6,12), ("Exp-C",7,12), ("Exp-F",9,8), ("Exp-F",12,8), ("Exp-F",13,8),
                               ("Exp-F",20,8), ("Exp-F",19,8), ("Exp-F",18,6), ("Exp-F",18,7), ("Exp-F",18,8)]}}
PH["eye"] = [EYE.get(k) for k in zip(PH.experiment, PH.tooth, PH.insp_order)]
PH["cls"] = np.select([PH.eye == "spall", PH.eye == "init", PH.tex >= 0.02], ["spall", "init", "ambiguous"], "healthy")
print(f"{len(PH):,} tooth-inspections from {int(PH.n_photos.sum()):,} photos:", PH.cls.value_counts().to_dict())

## 4 · Compute figure data

In [ ]:
# 4 · Compute D: gear table, channel stats, indicator metrics, trajectories, correlation, novelty prototype
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.stats import spearmanr
from sklearn.covariance import LedoitWolf

ORDER_G = ["Exp-A", "Exp-B", "Exp-C", "Exp-F", "Exp-G", "Exp-D", "Exp-E"]
EXPS = [g for g in ORDER_G if g in set(M1.experiment)]
BASE = (0.5, 12.0)                               # healthy baseline window (hours)
r3 = lambda x: None if x is None or (isinstance(x, float) and not np.isfinite(x)) else round(float(x), 3)
nums = lambda a, d=2: [None if v is None or not np.isfinite(v) else round(float(v), d) for v in a]

# runs and photo-inspection hours (a photo set is taken after each run; Break-In / Test-Start sit at 0 h)
runs = S1.groupby(["experiment", "run_num"]).agg(h0=("test_hours", "min"), h1=("test_hours", "max"),
                                                 temp=("ctx_temperature", "median")).reset_index()
runs["dup"] = (runs.experiment == "Exp-A") & (runs.run_num == 2)
runs["short"] = (runs.h1 - runs.h0) < 3
H1 = runs.set_index(["experiment", "run_num"]).h1
insp_hour = lambda e, o: 0.0 if o <= 0 else float(H1[(e, o)])
PH["hour"] = [insp_hour(e, o) for e, o in zip(PH.experiment, PH.insp_order)]

# gear table
split = M1.groupby("experiment").split.first()
gears = []
for g in EXPS:
    x, r, p = S1[S1.experiment == g], runs[runs.experiment == g], PH[PH.experiment == g]
    fo = p[p.eye.notna()].insp_order.min() if len(p) else np.nan
    gears.append(dict(gear=g[-1], split=split[g], runs=int(r.run_num.nunique()), end_h=r3(r.h1.max()),
        sensor_h=r3(x.ci_fm4.notna().sum() / 3600), steady_pct=r3(x.steady.mean() * 100),
        torque=r3(x.ctx_pau_torque.median()), temp=r3(x.ctx_temperature.median()), files=int(x.file.nunique()),
        rows=int(len(x)), short_runs=int(r.short.sum()), photo_insp=int(p.inspection.nunique()),
        photos=int(p.n_photos.sum()) if len(p) else 0,
        first_spall_insp=None if pd.isna(fo) else p[p.insp_order == fo].inspection.iloc[0],
        first_dmg_h=None if pd.isna(fo) else r3(insp_hour(g, int(fo)))))

# robust z per gear against its own baseline: (x - median) / (1.4826 * MAD)
Zs = {}
for g in EXPS:
    x = M1[M1.experiment == g].sort_values("hours"); b = x[(x.hours > BASE[0]) & (x.hours < BASE[1])]
    med = b[CI].median(); mad = (b[CI] - med).abs().median() * 1.4826
    z = (x[CI] - med) / mad.replace(0, np.nan); z["hours"] = x.hours.values; z["temp"] = x.ctx_temperature.values
    Zs[g] = z

# channel stats + Coble & Hines metrics
def mono(v):
    d = np.sign(np.diff(v)); d = d[d != 0]; return abs(d.sum()) / max(len(v) - 1, 1)
units = channels.set_index("column").unit
stats = []
for c in CI + CTX:
    v = M1[c].dropna()
    row = dict(col=c, unit=units.get(c), mean=r3(v.mean()), sd=r3(v.std()), p1=r3(v.quantile(.01)),
               p50=r3(v.median()), p99=r3(v.quantile(.99)), miss=r3(S1[c].isna().mean() * 100))
    if c in CI:
        mons, traj, ends, rhos, bmeds, bsds = [], [], [], [], [], []
        for g in EXPS:
            zz = Zs[g][[c, "hours"]].dropna()
            mons.append(mono(zz.groupby((zz.hours * 2).astype(int))[c].median().rolling(4, min_periods=1).median().values))
            traj.append(np.interp(np.linspace(0.05, 1, 40), (zz.hours / zz.hours.max()).values,
                                  zz[c].rolling(60, min_periods=10).median().bfill().values))
            ends.append(zz[zz.hours > zz.hours.max() - 1][c].median())
            x = M1[M1.experiment == g]; b = x[(x.hours > BASE[0]) & (x.hours < BASE[1])]
            rhos.append(abs(spearmanr(b[c], b.ctx_temperature, nan_policy="omit")[0]))
            bmeds.append(b[c].median()); bsds.append(b[c].std())
        cc = np.corrcoef(np.array(traj)); ends = np.array(ends)
        row.update(mono=r3(np.mean(mons)), trend=r3(np.nanmean(cc[np.triu_indices(len(traj), 1)])),
                   prog=r3(np.exp(-np.nanstd(ends) / np.nanmean(np.abs(ends)))), end_z=r3(np.nanmedian(ends)),
                   temp_rho=r3(np.nanmedian(rhos)), offset=r3(np.std(bmeds) / np.mean(bsds)))
    stats.append(row)

# 10-minute trajectories + DM4500 cumulative debris
PART["dm_total"] = PART[[f"dm4500_bin_{i}" for i in range(8)]].sum(axis=1)
PART["hours"] = PART.file.map(S1.groupby("file").test_hours.min())
traj = {}
for g in EXPS:
    a = Zs[g].groupby((Zs[g].hours * 6).astype(int)).median()
    d = PART[PART.experiment == g].dropna(subset=["hours"]); dk = d.groupby((d.hours * 6).astype(int)).dm_total.max()
    traj[g[-1]] = dict(h=nums(a.hours, 3), temp=nums(a.temp, 3), dm_h=nums(dk.index / 6, 3), dm=[int(v) for v in dk.values],
                       **{c: nums(a[c]) for c in CI})

# Spearman correlation of pooled 10-minute robust z, ordered by hierarchical clustering
P = pd.concat([Zs[g][CI].groupby((Zs[g].hours * 6).astype(int)).median() for g in EXPS]).dropna()
Cm = P.rank().corr().values
order = leaves_list(linkage(1 - np.abs(Cm[np.triu_indices(len(CI), 1)]), "average"))
corr = dict(cols=[CI[i] for i in order], m=[[round(float(Cm[i, j]), 2) for j in order] for i in order])

# Part B prototype: temperature-regressed, Ledoit-Wolf Mahalanobis novelty score + persistence onset rule
FEATS = ["ci_fm4", "ci_m6a", "ci_m8a", "ci_na4", "ci_e_ratio", "ci_kurtosis_tsa", "ci_rms_tsa", "ci_alr"]
def novelty(g, base_h=BASE, q=0.995, persist=60):
    g = g.dropna(subset=FEATS + ["ctx_temperature"])
    X = np.log(g[FEATS].abs() + 1e-9).values
    base = ((g.hours > base_h[0]) & (g.hours < base_h[1])).values
    T = np.c_[np.ones(len(g)), g.ctx_temperature]
    R = X - T @ np.linalg.lstsq(T[base], X[base], rcond=None)[0]
    Z = (R - R[base].mean(0)) / R[base].std(0)
    s = pd.Series(np.sqrt(LedoitWolf().fit(Z[base]).mahalanobis(Z) / len(FEATS)), index=g.hours.values)
    sm = s.rolling(30, min_periods=10).median()
    thr = float(np.quantile(sm[base].dropna(), q))
    above = (sm > thr) & (sm.index > base_h[1])
    run = above.astype(int).groupby((~above).cumsum()).cumsum(); hit = run[run >= persist]
    return sm, thr, (float(hit.index[0] - persist / 60) if len(hit) else None)

configs = {"base 0.5–6 h": (0.5, 6.0), "base 0.5–12 h": (0.5, 12.0), "base 0.5–18 h": (0.5, 18.0)}
nov, onsets = {}, {k: {} for k in configs}
for g in EXPS:
    for k, bh in configs.items():
        sm, thr, on = novelty(M1[M1.experiment == g], bh); onsets[k][g[-1]] = r3(on)
        if bh == BASE:
            q = sm.groupby((sm.index * 6).astype(int)).median()
            nov[g[-1]] = dict(h=nums(q.index / 6, 3), s=nums(q.values), thr=r3(thr), onset=r3(on))

# photo grid (tooth x inspection) and class balance
grid = {}
for g, p in PH.groupby("experiment"):
    ins = p.drop_duplicates("insp_order").sort_values("insp_order")
    key = p.set_index(["tooth", "insp_order"])
    cell = lambda t, o, col: (key.loc[(t, o), col] if (t, o) in key.index else None)
    grid[g[-1]] = dict(insp=ins.inspection.tolist(), hour=nums(ins.hour, 3),
                       eye=[[e if isinstance(e := cell(t, o, "eye"), str) else None for o in ins.insp_order] for t in range(1, 29)],
                       cells=[[r3(cell(t, o, "tex")) for o in ins.insp_order] for t in range(1, 29)])

# life split: hours to onset vs onset to end, with variance shares
END_H = runs.groupby("experiment").h1.max()
life = {}
for k in configs:
    rows = [dict(gear=g, init=round(on, 2), prop=round(END_H["Exp-" + g] - on, 2)) for g, on in onsets[k].items() if on is not None]
    I = np.array([r["init"] for r in rows]); Pp = np.array([r["prop"] for r in rows]); vt = (I + Pp).var()
    life[k] = dict(rows=rows, n=len(rows), var=dict(init=r3(I.var() / vt), prop=r3(Pp.var() / vt),
                                                    cov=r3(2 * np.cov(I, Pp, bias=True)[0, 1] / vt)))

D = dict(gears=gears, stats=stats, traj=traj, corr=corr, nov=nov, onsets=onsets, grid=grid, life=life,
         balance=PH.cls.value_counts().to_dict(),
         runs=[dict(gear=r.experiment[-1], run=int(r.run_num), h0=r3(r.h0), h1=r3(r.h1), temp=r3(r.temp),
                    dup=bool(r.dup), short=bool(r.short)) for r in runs.itertuples()],
         kpi=dict(rows=len(S1), files=int(S1.file.nunique()), photos=int(PH.n_photos.sum()),
                  tooth_insp=len(PH), minutes=len(M1)))
print("gears:", [g["gear"] for g in gears], "· onsets (0.5–12 h):", onsets["base 0.5–12 h"])

## 5 · Plot setup

In [ ]:
# 5 · Plot setup: shared Plotly styling (reads D from the Compute cell)
import json, math, plotly.graph_objects as go
from plotly.subplots import make_subplots
# D = json.load(open("data.json"))   # alternative: skip cells 1-4 and load the precomputed eda/out/data.json

GI    = {g["gear"]: g for g in D["gears"]}
GEARS = [g for g in ["A", "B", "C", "F", "G", "D", "E"] if g in GI]
SPLIT = {g: GI[g]["split"] for g in GEARS}
FIRST = {g: GI[g]["first_dmg_h"] for g in GEARS}      # None for gears without photos
END   = {g: GI[g]["end_h"] for g in GEARS}
RUNS  = pd.DataFrame(D["runs"])
NROW  = math.ceil(len(GEARS) / 2)                      # small-multiple grid rows

C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"                 # categorical slots 1-3 (validated)
Q = ["#e4e6e1", "#b7d3f6", "#3987e5", "#104281"]              # none / auto-flag / initiation / spall
INK, INK2, GRID, AXIS = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"

def style(fig, title, height=None):
    fig.update_layout(template="plotly_white", title=dict(text=title, x=0, font=dict(size=15, color=INK)),
                      font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", size=12, color=INK2),
                      margin=dict(l=60, r=20, t=90, b=40), hoverlabel=dict(bgcolor="white"),
                      legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0, traceorder="normal"))
    if height: fig.update_layout(height=height)
    fig.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=AXIS)
    fig.update_yaxes(gridcolor=GRID, zeroline=False)
    return fig

def slog(v):
    v = np.asarray(v, dtype=float); return np.sign(v) * np.log10(1 + np.abs(v))

def gapped(x, y, gap=0.5):
    """Insert NaN where time jumps > gap hours so lines don't bridge missing data."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    br = np.where(np.diff(x) > gap)[0] + 1
    return np.insert(x, br, np.nan), np.insert(y, br, np.nan)

DIAMOND = dict(symbol="diamond", size=11, color=INK, line=dict(color="white", width=2))
print(D["kpi"])

## Table 1: gear inventory

In [ ]:
# Table 1: one row per gear
t1 = pd.DataFrame(D["gears"])[["gear","split","runs","end_h","sensor_h","steady_pct","torque","temp",
                               "rows","short_runs","photo_insp","photos","first_spall_insp","first_dmg_h"]]
t1["gear"] = "Exp-" + t1.gear
t1.columns = ["Gear","Split","Runs","End of record (h)","Sensor hours","Steady %","Torque (ft·lbf)",
              "Oil temp (°F)","1-s rows","Short runs","Photo inspections","Photos","First visible damage","First damage (h)"]
t1.fillna({"First visible damage": "no photos"}).round(1)

## Figure 1: run timeline

In [ ]:
# Figure 1: runs on the cumulative operating-hour axis, photo inspections and first visible damage
r = RUNS.assign(dur=RUNS.h1 - RUNS.h0 - 0.2, start=RUNS.h0 + 0.1, y="Exp-" + RUNS.gear)
fig = go.Figure(go.Bar(
    y=r.y, x=r.dur, base=r.start, orientation="h", width=0.55, name="run (≈6.15 h)",
    marker=dict(color=[C1 if not d else "white" for d in r.dup],
                opacity=[0.4 if s else 1.0 for s in r.short],
                line=dict(color=[C2 if d else C1 for d in r.dup], width=[1.5 if d else 0 for d in r.dup]),
                pattern=dict(shape=["/" if d else "" for d in r.dup], fgcolor=C2)),
    customdata=np.c_[r.run, r.h0, r.h1, r.temp, np.where(r.dup, "yes (byte copy of Run-1)", "no")],
    hovertemplate="Run-%{customdata[0]}: %{customdata[1]:.1f}–%{customdata[2]:.1f} h<br>"
                  "median oil temp %{customdata[3]:.0f} °F<br>duplicate: %{customdata[4]}<extra>%{y}</extra>"))
for g, G in D["grid"].items():
    hrs = sorted(set(G["hour"]))
    fig.add_scatter(x=hrs, y=["Exp-"+g]*len(hrs), mode="markers", name="photo inspection",
                    marker=dict(symbol="line-ns-open", size=16, color=INK2, line=dict(width=1.5)),
                    showlegend=(g == "A"), hovertemplate="inspection at %{x:.1f} h<extra></extra>")
fd = [g for g in GEARS if FIRST[g] is not None]
fig.add_scatter(x=[FIRST[g] for g in fd], y=["Exp-"+g for g in fd], mode="markers", marker=DIAMOND,
                name="first visible damage", hovertemplate="first visible damage %{x:.1f} h<extra></extra>")
fig.update_yaxes(categoryorder="array", categoryarray=["Exp-"+g for g in GEARS][::-1])
fig.update_xaxes(title="cumulative operating hours", range=[0, 102])
style(fig, "Figure 1 · Run timeline (pale = short run <3 h, hatched = duplicate sensor data)", 400).show()

## Table 2: channel summary statistics

In [ ]:
# Table 2: pooled steady-state 1-minute medians, all 7 gears (missing % over all 1-s rows)
t2 = pd.DataFrame(D["stats"])[["col","unit","mean","sd","p1","p50","p99","miss"]]
t2.columns = ["Channel","Unit","Mean","SD","P1","Median","P99","Missing %"]
t2.fillna({"Unit": "–"}).style.format(precision=3, thousands=",").hide(axis="index")

## Rubric example photos (read from the bucket)

In [ ]:
# Rubric examples: no damage -> initiation -> spall, plus a shallow spall the texture score misses
RUBRIC = [("No visible damage", "Exp-B tooth 11 · after Run-5", "Exp-B_Run-5_All_Teeth_Tooth_11.jpg"),
          ("Initiation: small flake below the contact band", "Exp-B tooth 11 · after Run-6", "Exp-B_Run-6_All_Teeth_Tooth_11.jpg"),
          ("Spall: large crater across the flank", "Exp-B tooth 11 · after Run-7", "Exp-B_Run-7_All_Teeth_Tooth_11.jpg"),
          ("Shallow spall the texture score misses", "Exp-F tooth 19 · after Run-8", "Exp-F_Run-8_Tooth_19.jpg")]
imgs = []
for title, sub, name in RUBRIC:
    try:
        with fsspec.open(f"{PHOTO_DIR}/{name}", "rb") as f:
            im = Image.open(io.BytesIO(f.read())); w, h = im.size
            imgs.append((title, sub, np.asarray(im.crop((int(w * .12), int(h * .15), int(w * .88), int(h * .95))).reduce(4))))
    except (FileNotFoundError, OSError):
        print(f"  {name} not found under {PHOTO_DIR}")
if imgs:
    fig = make_subplots(rows=1, cols=len(imgs), subplot_titles=[f"{t}<br><sub>{s}</sub>" for t, s, _ in imgs],
                        horizontal_spacing=0.02)
    for i, (_, _, a) in enumerate(imgs, 1):
        fig.add_trace(go.Image(z=a, hoverinfo="skip"), 1, i)
    fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
    style(fig, "Tooth photo rubric examples", 340).update_annotations(font_size=11).show()

## Figure 2: tooth × inspection damage map and class balance

In [ ]:
# Figure 2: one heatmap per training gear; rows = teeth, columns = inspections
LAB = ["no visible damage", "auto-flagged, not reviewed", "initiation (eye-checked)", "spall (eye-checked)"]
cs  = [[0, Q[0]], [.25, Q[0]], [.25, Q[1]], [.5, Q[1]], [.5, Q[2]], [.75, Q[2]], [.75, Q[3]], [1, Q[3]]]
tg  = [g for g in ["A", "B", "C", "F"] if g in D["grid"]]
fig = make_subplots(rows=1, cols=len(tg), subplot_titles=[f"Exp-{g} · {len(D['grid'][g]['insp'])} inspections" for g in tg],
                    horizontal_spacing=0.04)
for i, g in enumerate(tg, 1):
    G = D["grid"][g]
    cells = np.array([[np.nan if v is None else v for v in row] for row in G["cells"]], dtype=float)
    eye = np.array([[e if isinstance(e, str) else "" for e in row] for row in G["eye"]], dtype=object)
    z = np.where(eye == "spall", 3, np.where(eye == "init", 2, np.where(np.nan_to_num(cells) >= 0.02, 1, 0))).astype(float)
    z[np.isnan(cells)] = np.nan
    x = [s.replace("Run-", "R").replace("Break-In", "Brk").replace("Test-Start", "Start").replace("Pre-Run", "Pre")
         for s in G["insp"]]
    cd = np.empty(z.shape + (2,), dtype=object)
    cd[..., 0] = np.where(np.isnan(cells), None, np.round(cells, 3))
    cd[..., 1] = [[LAB[int(k)] if not np.isnan(k) else "" for k in row] for row in z]
    fig.add_trace(go.Heatmap(z=z, x=x, y=list(range(1, 29)), zmin=-.5, zmax=3.5, colorscale=cs, showscale=False,
                             xgap=1.5, ygap=1.5, customdata=cd,
                             hovertemplate="tooth %{y} · %{x}<br>%{customdata[1]}<br>texture score %{customdata[0]}<extra></extra>"),
                  1, i)
    j = next((j for j in range(eye.shape[1]) if any(eye[:, j] != "")), None)
    if j is not None:
        fig.add_trace(go.Scatter(x=[x[j]], y=[-0.3], mode="markers", marker=DIAMOND, showlegend=False, cliponaxis=False,
                                 hovertemplate="first visible damage<extra></extra>"), 1, i)
for k, lab in enumerate(LAB):   # legend swatches
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=lab,
                             marker=dict(symbol="square", size=12, color=Q[k])), 1, 1)
fig.update_yaxes(autorange="reversed", showgrid=False, tickvals=list(range(1, 29, 3)))
fig.update_yaxes(title_text="tooth", col=1)
fig.update_xaxes(tickangle=-60, tickfont_size=10, showgrid=False)
style(fig, "Figure 2 · Tooth × inspection damage map (diamond = first visible damage)", 720)
fig.update_layout(legend=dict(y=-0.14, yanchor="top"), margin=dict(b=90)).show()

# Class balance across all tooth-inspections
b = D["balance"]; tot = sum(b.values())
bal = go.Figure()
for k, lab, c in [("healthy", LAB[0], Q[0]), ("ambiguous", LAB[1], Q[1]), ("init", LAB[2], Q[2]), ("spall", LAB[3], Q[3])]:
    v = b.get(k, 0)
    bal.add_bar(y=["tooth-inspections"], x=[v], orientation="h", name=f"{lab}: {v} ({100*v/tot:.1f}%)",
                marker=dict(color=c, line=dict(color="white", width=2)),
                hovertemplate=f"{lab}<br>{v} ({100*v/tot:.1f}%)<extra></extra>")
bal.update_layout(barmode="stack", yaxis_visible=False, xaxis_title="count")
style(bal, f"Class balance, all {tot:,} tooth-inspections", 240).show()

## Figure 3: condition-indicator trajectories
Change `CI` (or call `plot_traj("ci_fm4")`) to switch indicators; use `"dm"` for the DM4500 oil-debris counter.

In [ ]:
# Figure 3: robust z vs each gear's own 0.5-12 h baseline, signed-log y, shared axes.
# Change CI to any key in D["traj"]["A"] (e.g. "ci_fm4", "ci_m6a", "ci_na4"), or "dm" for DM4500 debris count.
CI = "ci_m8a"

def plot_traj(ci=CI):
    fig = make_subplots(rows=NROW, cols=2, shared_xaxes="all", shared_yaxes="all", vertical_spacing=0.06,
                        subplot_titles=[f"Exp-{g} · {SPLIT[g]}" for g in GEARS])
    ys = {}
    for g in GEARS:
        T = D["traj"][g]
        if ci == "dm":
            ys[g] = (np.asarray(T["dm_h"], float), np.asarray(T["dm"], float), np.asarray(T["dm"], float))
        else:
            raw = np.asarray([np.nan if v is None else v for v in T[ci]], float)
            ys[g] = (np.asarray(T["h"], float), raw, slog(raw))
    top = np.nanmax([np.nanmax(v[2]) for v in ys.values()])
    for i, g in enumerate(GEARS):
        row, col = i // 2 + 1, i % 2 + 1
        x, raw, y = ys[g]
        xg, yg = gapped(x, y); _, rg = gapped(x, raw)
        fig.add_trace(go.Scatter(x=xg, y=yg, mode="lines", line=dict(color=C1, width=2), customdata=rg,
                                 showlegend=False, hovertemplate="%{x:.1f} h<br>value %{customdata:.2f}<extra>Exp-" + g + "</extra>"),
                      row, col)
        for h0 in RUNS[(RUNS.gear == g) & (RUNS.run > 1)].h0:   # run boundaries
            fig.add_shape(type="line", x0=h0, x1=h0, y0=0, y1=0.06, yref="y domain" if i == 0 else f"y{i+1} domain",
                          xref="x" if i == 0 else f"x{i+1}", line=dict(color=AXIS, width=1))
        if FIRST[g] is not None:
            fig.add_trace(go.Scatter(x=[FIRST[g]], y=[top], mode="markers", marker=DIAMOND, showlegend=False,
                                     hovertemplate="first visible damage %{x:.1f} h<extra></extra>"), row, col)
    if ci != "dm":
        lo = int(np.floor(np.nanmin([np.nanmin(v[2]) for v in ys.values()]))); tv = np.arange(lo, int(np.ceil(top)) + 1)
        fig.update_yaxes(tickvals=tv, ticktext=[("−" if t < 0 else "") + ("0" if t == 0 else "10" if abs(t) == 1 else f"10<sup>{abs(t)}</sup>") for t in tv])
    fig.update_xaxes(range=[0, 100]); fig.update_xaxes(title_text="cumulative hours", row=NROW)
    ylab = "DM4500 cumulative count" if ci == "dm" else "robust z (signed log)"
    name = "DM4500 debris" if ci == "dm" else ci.replace("ci_", "").upper()
    style(fig, f"Figure 3 · {name} trajectory per gear (diamond = first visible damage; ticks = run boundaries)", 60 + 225 * NROW)
    fig.update_yaxes(title_text=ylab, col=1).show()

plot_traj(CI)

## Table 3: indicator quality (Coble & Hines metrics)

In [ ]:
# Table 3: trendability, monotonicity, prognosability, end-of-life z, temperature sensitivity, between-gear offset
t3 = (pd.DataFrame([s for s in D["stats"] if s["col"].startswith("ci_")])
        [["col","trend","mono","prog","end_z","temp_rho","offset"]]
        .assign(col=lambda d: d.col.str.replace("ci_", "").str.upper())
        .rename(columns={"col":"Indicator","trend":"Trendability","mono":"Monotonicity","prog":"Prognosability",
                         "end_z":"End z (median)","temp_rho":"Temp |ρ|","offset":"Gear offset (SD)"})
        .sort_values("Trendability", ascending=False).reset_index(drop=True))
t3.style.format(precision=2).bar(subset=["Trendability","Monotonicity","Prognosability"], color="#b7d3f6", vmin=0, vmax=1).hide(axis="index")

## Figure 4: indicator redundancy

In [ ]:
# Figure 4: Spearman correlation of per-gear robust z (10-min), clustered order
labs = [c.replace("ci_", "").upper() for c in D["corr"]["cols"]]
fig = go.Figure(go.Heatmap(z=D["corr"]["m"], x=labs, y=labs, zmin=-1, zmax=1, xgap=1, ygap=1,
                           colorscale=[[0, C1], [.5, "#f0efec"], [1, "#e34948"]],
                           colorbar=dict(title="ρ", thickness=12),
                           hovertemplate="%{y} × %{x}<br>ρ = %{z:.2f}<extra></extra>"))
fig.update_yaxes(autorange="reversed", scaleanchor="x", showgrid=False); fig.update_xaxes(tickangle=-60, showgrid=False)
style(fig, "Figure 4 · Indicator redundancy (Spearman ρ): pick one or two CIs per block", 780).show()

## Figure 5: novelty score per gear (Part B prototype)

In [ ]:
# Figure 5: Mahalanobis novelty score (baseline 0.5-12 h), threshold, onset, first visible damage
fig = make_subplots(rows=NROW, cols=2, shared_xaxes="all", shared_yaxes="all", vertical_spacing=0.06,
                    subplot_titles=[f"Exp-{g} · {SPLIT[g]}" for g in GEARS])
for i, g in enumerate(GEARS):
    row, col = i // 2 + 1, i % 2 + 1; N = D["nov"][g]; first = i == 0
    x, s = gapped(N["h"], [np.nan if v is None else v for v in N["s"]])
    fig.add_trace(go.Scatter(x=x, y=s, mode="lines", line=dict(color=C1, width=2), name="novelty score",
                             showlegend=first, hovertemplate="%{x:.1f} h · score %{y:.2f}<extra>Exp-" + g + "</extra>"), row, col)
    fig.add_trace(go.Scatter(x=[0, END[g]], y=[N["thr"]]*2, mode="lines", line=dict(color=C2, width=1.5),
                             name="alarm threshold (99.5th pct of baseline)", showlegend=first,
                             hovertemplate="threshold %{y:.2f}<extra></extra>"), row, col)
    if N["onset"] is not None:
        fig.add_trace(go.Scatter(x=[N["onset"]]*2, y=[0.4, 150], mode="lines", line=dict(color=INK, width=1.5),
                                 name="detected onset", showlegend=first, hovertemplate=f"onset {N['onset']:.1f} h<extra></extra>"), row, col)
    if FIRST[g] is not None:
        fig.add_trace(go.Scatter(x=[FIRST[g]], y=[120], mode="markers", marker=DIAMOND, name="first visible damage",
                                 showlegend=(g == "A"), hovertemplate="first visible damage %{x:.1f} h<extra></extra>"), row, col)
fig.update_yaxes(type="log", range=[np.log10(0.4), np.log10(150)], tickvals=[0.5, 1, 2, 5, 10, 20, 50, 100],
                 ticktext=["0.5", "1", "2", "5", "10", "20", "50", "100"])
fig.update_xaxes(range=[0, 100]); fig.update_xaxes(title_text="cumulative hours", row=NROW)
style(fig, "Figure 5 · Novelty score per gear (√(D²/p), 30-min median, log y)", 60 + 225 * NROW).show()

## Figure 6: onset stability across baseline windows

In [ ]:
# Figure 6: onset per gear under three baseline windows vs first visible photo damage
CFG = list(D["onsets"]); cols = [C1, C2, C3]
ypos = {g: len(GEARS) - 1 - i for i, g in enumerate(GEARS)}
fig = go.Figure()
for g in GEARS:
    fig.add_shape(type="line", x0=0, x1=END[g], y0=ypos[g], y1=ypos[g], line=dict(color=AXIS, width=2), layer="below")
    fig.add_shape(type="line", x0=END[g], x1=END[g], y0=ypos[g]-.25, y1=ypos[g]+.25, line=dict(color=INK2, width=2))
for j, k in enumerate(CFG):
    gs = [g for g in GEARS if D["onsets"][k][g] is not None]
    fig.add_scatter(x=[D["onsets"][k][g] for g in gs], y=[ypos[g] + (j-1)*0.15 for g in gs], mode="markers",
                    name=f"onset, {k}", marker=dict(size=10, color=cols[j], line=dict(color="white", width=2)),
                    customdata=gs, hovertemplate="Exp-%{customdata}: onset %{x:.1f} h<extra>" + k + "</extra>")
fd = [g for g in GEARS if FIRST[g] is not None]
fig.add_scatter(x=[FIRST[g] for g in fd], y=[ypos[g] for g in fd], mode="markers", marker=DIAMOND,
                name="first visible photo damage", hovertemplate="first visible damage %{x:.1f} h<extra></extra>")
fig.update_yaxes(tickvals=list(ypos.values()), ticktext=["Exp-"+g for g in ypos], range=[-.6, len(GEARS)-.4], showgrid=False)
fig.update_xaxes(title="cumulative hours (tick = end of record)", range=[0, 102])
style(fig, "Figure 6 · Onset stability across baseline windows", 440).show()

## Figure 7: life split into initiation and propagation, with variance shares

In [ ]:
# Figure 7: hours to onset (initiation) + onset to end (propagation), sorted by total life
CFG = list(D["onsets"])
photo = [dict(gear=g, init=FIRST[g], prop=END[g]-FIRST[g]) for g in GEARS if FIRST[g] is not None]
sensor = D["life"][CFG[1]]["rows"]                     # baseline 0.5-12 h
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=["Training gears · onset = first visible photo damage",
                                    f"All gears · onset = sensor prototype ({CFG[1]})"])
for c, rows in enumerate([photo, sensor], 1):
    rows = sorted(rows, key=lambda r: r["init"] + r["prop"])
    y = ["Exp-" + r["gear"] for r in rows]
    fig.add_bar(y=y, x=[r["init"] for r in rows], orientation="h", name="hours to onset (initiation)",
                marker=dict(color=C1, line=dict(color="white", width=2)), width=0.6, showlegend=(c == 1),
                hovertemplate="%{y}: %{x:.1f} h to onset<extra></extra>", row=1, col=c)
    fig.add_bar(y=y, x=[r["prop"] for r in rows], orientation="h", name="hours from onset to end (propagation)",
                marker=dict(color=C2, line=dict(color="white", width=2)), width=0.6, showlegend=(c == 1),
                text=[f"{r['init']+r['prop']:.0f} h" for r in rows], textposition="outside", cliponaxis=False,
                hovertemplate="%{y}: %{x:.1f} h onset to end<extra></extra>", row=1, col=c)
fig.update_layout(barmode="stack"); fig.update_xaxes(range=[0, 110], title_text="hours"); fig.update_yaxes(showgrid=False)
style(fig, "Figure 7 · Life split into initiation and propagation (longest life at top)", 460)
fig.update_layout(legend=dict(y=-0.2, yanchor="top")).show()

# Share of var(total life): var(I) + var(P) + 2·cov(I,P) = 1
def shares(rows):
    I = np.array([r["init"] for r in rows]); P = np.array([r["prop"] for r in rows]); vt = (I + P).var()
    return dict(n=len(rows), init=I.var()/vt, prop=P.var()/vt, cov=2*np.cov(I, P, bias=True)[0, 1]/vt)
vt = pd.DataFrame([dict(onset="Photo, first visible damage", **shares(photo))] +
                  [dict(onset=f"Sensor prototype, {k}", n=D["life"][k]["n"], **D["life"][k]["var"]) for k in CFG])
vt["largest"] = vt[["init","prop","cov"]].idxmax(axis=1).map({"init":"initiation","prop":"propagation","cov":"covariance"})
vt.rename(columns={"onset":"Onset definition","n":"Gears","init":"var(I) share","prop":"var(P) share",
                   "cov":"2·cov share","largest":"Largest share"}).round(2)